In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/q17_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Cell 2 optimized for cudf

# 1. Filter PART to get the relevant keys on GPU using boolean indexing instead of .query
filtered_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

# 2. Early‐filter LINEITEM to just those keys, then compute the 20%‐scaled average quantity per key on GPU
li_subset = lineitem[lineitem.L_PARTKEY.isin(filtered_keys)]
avg_map = li_subset.groupby('L_PARTKEY')['L_QUANTITY'].mean() * 0.2

# 3. Merge the precomputed avg back into the subset (avoiding .map) and select only the needed cols
li_with_avg = li_subset[['L_PARTKEY', 'L_QUANTITY', 'L_EXTENDEDPRICE']]  
li_with_avg = li_with_avg.merge(avg_map.rename('avg'), 
                                 left_on='L_PARTKEY', 
                                 right_index=True)

# 4. Apply the quantity filter and compute the final metric on GPU
filtered = li_with_avg[li_with_avg.L_QUANTITY < li_with_avg.avg]
total = pd.DataFrame({
    'avg_yearly': [filtered.L_EXTENDEDPRICE.sum() / 7.0]
})